# Supervised Learning. Classification - Don't Get Kicked

Задача: по характеристикам подержанной машины с аукциона предсказать, окажется ли
она «лимоном» (`IsBadBuy = 1`) - то есть плохой покупкой. Классическая бинарная
классификация с сильным дисбалансом классов (плохих покупок примерно 12%).

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from category_encoders import CountEncoder
from sklearn.metrics import roc_auc_score, average_precision_score

pd.set_option("display.max_columns", 60)
RND = 42

## 1. Загрузка данных

Соревнование Don't Get Kicked. У файла `test.csv` от Kaggle нет колонки `IsBadBuy`
(это лидербордный тест без таргета), поэтому для обучения и оценки качества
используется с размеченный `training.csv`.

In [2]:
df = pd.read_csv("data/training.csv")
df["PurchDate"] = pd.to_datetime(df["PurchDate"])

print("shape:", df.shape)
print("\nбаланс таргета:")
print(df["IsBadBuy"].value_counts(normalize=True).round(4))
df.head(3)

shape: (72983, 34)

баланс таргета:
IsBadBuy
0    0.877
1    0.123
Name: proportion, dtype: float64


,RefId,IsBadBuy,PurchDate,Auction,VehYear,VehicleAge,Make,Model,Trim,SubModel,Color,Transmission,WheelTypeID,WheelType,VehOdo,Nationality,Size,TopThreeAmericanName,MMRAcquisitionAuctionAveragePrice,MMRAcquisitionAuctionCleanPrice,MMRAcquisitionRetailAveragePrice,MMRAcquisitonRetailCleanPrice,MMRCurrentAuctionAveragePrice,MMRCurrentAuctionCleanPrice,MMRCurrentRetailAveragePrice,MMRCurrentRetailCleanPrice,PRIMEUNIT,AUCGUART,BYRNO,VNZIP1,VNST,VehBCost,IsOnlineSale,WarrantyCost
0,1,0,2009-12-07,ADESA,2006,3,MAZDA,MAZDA3,i,4D SEDAN I,RED,AUTO,1.0,Alloy,89046,OTHER ASIAN,MEDIUM,OTHER,8155.0,9829.0,11636.0,13600.0,7451.0,8552.0,11597.0,12409.0,NaN,NaN,21973,33619,FL,7100.0,0,1113
1,2,0,2009-12-07,ADESA,2004,5,DODGE,1500 RAM PICKUP 2WD,ST,QUAD CAB 4.7L SLT,WHITE,AUTO,1.0,Alloy,93593,AMERICAN,LARGE TRUCK,CHRYSLER,6854.0,8383.0,10897.0,12572.0,7456.0,9222.0,11374.0,12791.0,NaN,NaN,19638,33619,FL,7600.0,0,1053
2,3,0,2009-12-07,ADESA,2005,4,DODGE,STRATUS V6,SXT,4D SEDAN SXT FFV,MAROON,AUTO,2.0,Covers,73807,AMERICAN,MEDIUM,CHRYSLER,3202.0,4760.0,6943.0,8457.0,4035.0,5557.0,7146.0,8702.0,NaN,NaN,19638,33619,FL,4900.0,0,1389


In [3]:
# много пропусков, особенно PRIMEUNIT/AUCGUART почти пустые
na = df.isna().sum()
na[na > 0].sort_values(ascending=False)

AUCGUART                             69564
PRIMEUNIT                            69564
WheelType                             3174
WheelTypeID                           3169
Trim                                  2360
MMRCurrentAuctionAveragePrice          315
MMRCurrentAuctionCleanPrice            315
MMRCurrentRetailAveragePrice           315
MMRCurrentRetailCleanPrice             315
MMRAcquisitionAuctionCleanPrice         18
MMRAcquisitionAuctionAveragePrice       18
MMRAcquisitionRetailAveragePrice        18
MMRAcquisitonRetailCleanPrice           18
Transmission                             9
SubModel                                 8
Color                                    8
Size                                     5
Nationality                              5
TopThreeAmericanName                     5
dtype: int64

## 2. Сплит train / valid / test по дате

По условию: `train.PurchDate < valid.PurchDate < test.PurchDate`. Первую треть дат
в train, среднюю - в valid, последнюю - в test. 


In [4]:
q1, q2 = df["PurchDate"].quantile([1/3, 2/3])

train = df[df["PurchDate"] <= q1].copy()
valid = df[(df["PurchDate"] > q1) & (df["PurchDate"] <= q2)].copy()
test  = df[df["PurchDate"] > q2].copy()

for name, part in [("train", train), ("valid", valid), ("test", test)]:
    print(f"{name:5s} | n={len(part):6d} | "
          f"{part.PurchDate.min().date()} .. {part.PurchDate.max().date()} | "
          f"bad rate={part.IsBadBuy.mean():.3f}")

# проверка непересечения по времени
assert train.PurchDate.max() < valid.PurchDate.min() < valid.PurchDate.max() < test.PurchDate.min()

train | n= 24391 | 2009-01-05 .. 2009-09-15 | bad rate=0.115
valid | n= 24327 | 2009-09-16 .. 2010-05-14 | bad rate=0.131
test  | n= 24265 | 2010-05-17 .. 2010-12-30 | bad rate=0.124


## 3. Препроцессинг фичей

- числовые колонки остаются как есть, пропуски заполняются медианой по train;
- идентификаторы-коды (`WheelTypeID`, `BYRNO`, `VNZIP1`) по смыслу не числа, а категории;
- категориальные кодируются **count-кодированием** через `CountEncoder` из библиотеки
  `category_encoders`

In [5]:
TARGET = "IsBadBuy"
DROP = ["RefId", "IsBadBuy", "PurchDate"]
ID_LIKE = ["WheelTypeID", "BYRNO", "VNZIP1"]   # числовые по типу, но это коды

num_cols = [c for c in df.columns
            if c not in DROP and pd.api.types.is_numeric_dtype(df[c]) and c not in ID_LIKE]
cat_cols = [c for c in df.columns
            if c not in DROP and (not pd.api.types.is_numeric_dtype(df[c]) or c in ID_LIKE)]

print("числовых:", len(num_cols), num_cols)
print("\nкатегориальных:", len(cat_cols), cat_cols)

числовых: 14 ['VehYear', 'VehicleAge', 'VehOdo', 'MMRAcquisitionAuctionAveragePrice', 'MMRAcquisitionAuctionCleanPrice', 'MMRAcquisitionRetailAveragePrice', 'MMRAcquisitonRetailCleanPrice', 'MMRCurrentAuctionAveragePrice', 'MMRCurrentAuctionCleanPrice', 'MMRCurrentRetailAveragePrice', 'MMRCurrentRetailCleanPrice', 'VehBCost', 'IsOnlineSale', 'WarrantyCost']

категориальных: 17 ['Auction', 'Make', 'Model', 'Trim', 'SubModel', 'Color', 'Transmission', 'WheelTypeID', 'WheelType', 'Nationality', 'Size', 'TopThreeAmericanName', 'PRIMEUNIT', 'AUCGUART', 'BYRNO', 'VNZIP1', 'VNST']


In [6]:
# официальный CountEncoder из category_encoders (ссылка из README)
# учим только на train; новые категории -> 0, NaN считаем как отдельную категорию
count_enc = CountEncoder(cols=cat_cols, handle_unknown=0, handle_missing="count")
count_enc.fit(train[cat_cols])

def make_features(part, extra_num=None):
    "Числовые фичи + count-кодированные категориальные (энкодер обучен на train)."
    extra_num = extra_num or []
    X = pd.DataFrame(index=part.index)
    for c in num_cols + extra_num:
        X[c] = part[c]
    enc = count_enc.transform(part[cat_cols])[cat_cols]
    enc.columns = [c + "_count" for c in cat_cols]
    return pd.concat([X, enc], axis=1)

def prepare(train_part, *other_parts, extra_num=None):
    "Фичи -> импутация медианой train -> стандартизация. Скейлер учим на train."
    Xtr = make_features(train_part, extra_num)
    med = Xtr.median()
    Xtr = Xtr.fillna(med)
    scaler = StandardScaler().fit(Xtr)

    out = [scaler.transform(Xtr)]
    for part in other_parts:
        Xo = make_features(part, extra_num).fillna(med)
        out.append(scaler.transform(Xo))
    return out, scaler, list(Xtr.columns)

(Xtr, Xva), scaler, feat_names = prepare(train, valid)
ytr = train[TARGET].values
yva = valid[TARGET].values
print("матрица train:", Xtr.shape, "| фичей:", len(feat_names))

матрица train: (24391, 31) | фичей: 31


## 4. Базовые модели sklearn: LogisticRegression, GaussianNB, KNN

Обучение трех моделей на train, измерение Gini на valid.

In [ ]:
def sk_gini(y_true, scores):
    return abs(2 * roc_auc_score(y_true, scores) - 1)

baseline_models = {
    "LogisticRegression": LogisticRegression(max_iter=2000, random_state=RND),
    "GaussianNB":         GaussianNB(),
    "KNN (k=50)":         KNeighborsClassifier(n_neighbors=50),
}

baseline_gini = {}
for name, model in baseline_models.items():
    model.fit(Xtr, ytr)
    proba = model.predict_proba(Xva)[:, 1]
    baseline_gini[name] = sk_gini(yva, proba)
    print(f"{name:20s} valid Gini = {baseline_gini[name]:.4f}")

best_name = max(baseline_gini, key=baseline_gini.get)
print(f"\nлучшая: {best_name} ({baseline_gini[best_name]:.4f}) - порог 0.15 пройден")

LogisticRegression   valid Gini = 0.4543
GaussianNB           valid Gini = 0.4497
KNN (k=50)           valid Gini = 0.4285

лучшая: LogisticRegression (0.4543) — порог 0.15 пройден


Лучше всех Logistic Regression.
GaussianNB чуть слабее,
потому что его предположение о независимости признаков и нормальности грубо нарушается -
определенные фичи сильно скоррелированы между собой. KNN страдает от размерности (17+ признаков)
и неоднородности шкал: евклидово расстояние в таком пространстве уже не очень информативно,
плюс классы сильно несбалансированы.

## 5. Своя реализация Gini

Gini рассчитывается как `|2*ROC AUC - 1|`, а ROC AUC реализую сам.

ROC AUC = вероятность того, что случайный объект класса 1 получит больший скор, чем
случайный объект класса 0. Рассчитывается как статистика Манна–Уитни.

In [8]:
def my_roc_auc(y_true, scores):
    y_true = np.asarray(y_true)
    scores = np.asarray(scores, dtype=float)
    n_pos = y_true.sum()
    n_neg = len(y_true) - n_pos
    if n_pos == 0 or n_neg == 0:
        return 0.5

    # средние ранги (1..n), чтобы корректно учесть одинаковые скоры
    order = np.argsort(scores, kind="mergesort")
    ranks = np.empty(len(scores), dtype=float)
    sorted_scores = scores[order]
    i = 0
    while i < len(scores):
        j = i
        while j + 1 < len(scores) and sorted_scores[j + 1] == sorted_scores[i]:
            j += 1
        ranks[order[i:j + 1]] = (i + j) / 2.0 + 1.0   # средний ранг группы
        i = j + 1

    sum_pos = ranks[y_true == 1].sum()
    # формула Манна-Уитни
    auc = (sum_pos - n_pos * (n_pos + 1) / 2.0) / (n_pos * n_neg)
    return auc

def my_gini(y_true, scores):
    return abs(2 * my_roc_auc(y_true, scores) - 1)

# сверка с sklearn
proba = baseline_models[best_name].predict_proba(Xva)[:, 1]
print("my  ROC AUC:", round(my_roc_auc(yva, proba), 6))
print("skl ROC AUC:", round(roc_auc_score(yva, proba), 6))
print("my  Gini   :", round(my_gini(yva, proba), 6))
print("skl Gini   :", round(sk_gini(yva, proba), 6))
assert np.isclose(my_roc_auc(yva, proba), roc_auc_score(yva, proba), atol=1e-6)

my  ROC AUC: 0.727145
skl ROC AUC: 0.727145
my  Gini   : 0.45429
skl Gini   : 0.45429


## 6. Свои реализации LogisticRegression, KNN, NaiveBayes





### 6.1 Логистическая регрессия на SGD

In [9]:
def _sigmoid(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -30, 30)))

class MyLogisticRegression:
    def __init__(self, lr=0.5, epochs=40, batch_size=512, l2=0.0, random_state=42):
        self.lr = lr
        self.epochs = epochs
        self.batch_size = batch_size
        self.l2 = l2            # L2-регуляризация (0 = выключена)
        self.random_state = random_state

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float)
        rng = np.random.default_rng(self.random_state)
        n, d = X.shape
        self.w = np.zeros(d)
        self.b = 0.0

        for _ in range(self.epochs):
            idx = rng.permutation(n)
            for s in range(0, n, self.batch_size):
                bi = idx[s:s + self.batch_size]
                xb, yb = X[bi], y[bi]
                pred = _sigmoid(xb @ self.w + self.b)
                err = pred - yb                      # (предсказание - факт)
                grad_w = xb.T @ err / len(bi) + self.l2 * self.w
                grad_b = err.mean()
                self.w -= self.lr * grad_w
                self.b -= self.lr * grad_b
        return self

    def predict_proba(self, X):
        return _sigmoid(np.asarray(X, dtype=float) @ self.w + self.b)

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int)

### 6.2 Наивный Байес (гауссовский)

In [10]:
class MyGaussianNB:
    def __init__(self, eps=1e-9):
        self.eps = eps   

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y)
        self.classes_ = np.unique(y)
        self.mean_, self.var_, self.prior_ = {}, {}, {}
        for c in self.classes_:
            Xc = X[y == c]
            self.mean_[c] = Xc.mean(axis=0)
            self.var_[c] = Xc.var(axis=0) + self.eps
            self.prior_[c] = len(Xc) / len(X)
        return self

    def _log_posterior(self, X):
        # log p(y=c) + sum_d log N(x_d | mu, var)
        cols = []
        for c in self.classes_:
            log_prior = np.log(self.prior_[c])
            ll = -0.5 * (np.log(2 * np.pi * self.var_[c])
                         + (X - self.mean_[c]) ** 2 / self.var_[c])
            cols.append(log_prior + ll.sum(axis=1))
        return np.column_stack(cols)

    def predict_proba(self, X):
        X = np.asarray(X, dtype=float)
        logp = self._log_posterior(X)
        logp -= logp.max(axis=1, keepdims=True)      
        p = np.exp(logp)
        p /= p.sum(axis=1, keepdims=True)
        return p[:, 1]                               # вероятность класса 1

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int)

### 6.3 KNN

In [11]:
class MyKNN:
    def __init__(self, k=50, batch_size=1000):
        self.k = k
        self.batch_size = batch_size

    def fit(self, X, y):
        self.X = np.asarray(X, dtype=float)
        self.y = np.asarray(y)
        return self

    def predict_proba(self, X):
        X = np.asarray(X, dtype=float)
        train_sq = (self.X ** 2).sum(axis=1)          # ||t||^2
        out = np.empty(len(X))
        for s in range(0, len(X), self.batch_size):
            xb = X[s:s + self.batch_size]
            # ||x-t||^2 = ||x||^2 + ||t||^2 - 2 x·t
            d2 = (xb ** 2).sum(axis=1, keepdims=True) + train_sq - 2 * xb @ self.X.T
            nn = np.argpartition(d2, self.k, axis=1)[:, :self.k]
            out[s:s + len(xb)] = self.y[nn].mean(axis=1)
        return out

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int)

### Сравнение своих моделей с sklearn


In [12]:
# своя логрег
my_lr = MyLogisticRegression(lr=0.5, epochs=40, batch_size=512).fit(Xtr, ytr)
g_lr = my_gini(yva, my_lr.predict_proba(Xva))

# свой NB
my_nb = MyGaussianNB().fit(Xtr, ytr)
g_nb = my_gini(yva, my_nb.predict_proba(Xva))

# свой KNN
rng = np.random.default_rng(RND)
my_knn = MyKNN(k=50).fit(Xtr, ytr)
g_knn = my_gini(yva, my_knn.predict_proba(Xva))

print(f"{'модель':22s}{'своя':>10s}{'sklearn':>12s}")
print(f"{'LogisticRegression':22s}{g_lr:>10.4f}{baseline_gini['LogisticRegression']:>12.4f}")
print(f"{'GaussianNB':22s}{g_nb:>10.4f}{baseline_gini['GaussianNB']:>12.4f}")
print(f"{'KNN':22s}{g_knn:>10.4f}{baseline_gini['KNN (k=50)']:>12.4f}")

модель                      своя     sklearn
LogisticRegression        0.4513      0.4543
GaussianNB                0.4497      0.4497
KNN                       0.4285      0.4285


## 7. Нелинейные признаки

Новые фичи:

- **отношения цен**: за сколько купили относительно рыночной оценки (`VehBCost / MMR...`),
  спред розница/аукцион, падение текущей цены к закупочной;
- **odo_per_year** - пробег на год возраста (агрессивность эксплуатации);
- **warranty_to_cost** - стоимость гарантии относительно цены машины;
- **groupby-фичи**: средняя цена по модели и средний пробег по марке (учим агрегаты на train).

In [13]:
def add_nonlinear(part):
    p = part.copy()
    p["cost_to_mmr"]      = p["VehBCost"] / (p["MMRAcquisitionAuctionAveragePrice"] + 1)
    p["retail_to_auction"] = p["MMRAcquisitionRetailAveragePrice"] / (p["MMRAcquisitionAuctionAveragePrice"] + 1)
    p["curr_to_acq"]      = p["MMRCurrentAuctionAveragePrice"] / (p["MMRAcquisitionAuctionAveragePrice"] + 1)
    p["odo_per_year"]     = p["VehOdo"] / (p["VehicleAge"] + 1)
    p["warranty_to_cost"] = p["WarrantyCost"] / (p["VehBCost"] + 1)
    return p

# groupby-фичи считаем на train и переносим на valid/test
model_mean_cost = train.groupby("Model")["VehBCost"].mean()
make_mean_odo   = train.groupby("Make")["VehOdo"].mean()
glob_cost, glob_odo = train["VehBCost"].mean(), train["VehOdo"].mean()

def add_groupby(part):
    p = part.copy()
    p["model_mean_cost"] = p["Model"].map(model_mean_cost).fillna(glob_cost)
    p["make_mean_odo"]   = p["Make"].map(make_mean_odo).fillna(glob_odo)
    return p

EXTRA = ["cost_to_mmr", "retail_to_auction", "curr_to_acq", "odo_per_year",
         "warranty_to_cost", "model_mean_cost", "make_mean_odo"]

train_fe = add_groupby(add_nonlinear(train))
valid_fe = add_groupby(add_nonlinear(valid))
test_fe  = add_groupby(add_nonlinear(test))

(Xtr2, Xva2), scaler2, feat_names2 = prepare(train_fe, valid_fe, extra_num=EXTRA)
print("стало фичей:", len(feat_names2), "(было", len(feat_names), ")")

стало фичей: 38 (было 31 )


In [14]:
# переобучаем три модели на расширенном наборе
fe_models = {
    "LogisticRegression": LogisticRegression(max_iter=2000, random_state=RND),
    "GaussianNB":         GaussianNB(),
    "KNN (k=50)":         KNeighborsClassifier(n_neighbors=50),
}
print(f"{'модель':22s}{'было':>9s}{'стало':>9s}")
fe_gini = {}
for name, model in fe_models.items():
    model.fit(Xtr2, ytr)
    fe_gini[name] = sk_gini(yva, model.predict_proba(Xva2)[:, 1])
    base_key = name if name in baseline_gini else "KNN (k=50)"
    print(f"{name:22s}{baseline_gini[name]:>9.4f}{fe_gini[name]:>9.4f}")

print(f"\nGini лог.регрессии: {baseline_gini['LogisticRegression']:.4f} -> "
      f"{fe_gini['LogisticRegression']:.4f}  (+{fe_gini['LogisticRegression']-baseline_gini['LogisticRegression']:.4f})")

модель                     было    стало
LogisticRegression       0.4543   0.4565
GaussianNB               0.4497   0.4404
KNN (k=50)               0.4285   0.4292

Gini лог.регрессии: 0.4543 -> 0.4565  (+0.0022)


Прирост есть, хоть и небольшой. Наивный байес показал ухудшенный результат, т. к. степень коррелированности фичей увеличилась.

## 8. Отбор признаков: руками vs L1-регуляризация

Дальше два способа выкинуть
бесполезные фичи:

1. **руками** - убрать признаки с самыми маленькими по модулю коэффициентами;
2. **L1-регуляризация** (`penalty='l1'`) - зануляет слабые веса автоматически.

In [15]:
lr_full = LogisticRegression(max_iter=2000, random_state=RND).fit(Xtr2, ytr)
coef = pd.Series(lr_full.coef_[0], index=feat_names2).sort_values(key=np.abs, ascending=False)
print("топ-12 признаков по |коэффициенту|:")
print(coef.head(12).round(3))
print("\nслабейшие признаки (кандидаты на выброс):")
print(coef.tail(8).round(3))

топ-12 признаков по |коэффициенту|:
WheelTypeID_count              -0.289
WheelType_count                -0.289
TopThreeAmericanName_count     -0.282
VehBCost                       -0.276
VehOdo                          0.259
Nationality_count               0.212
Model_count                    -0.191
VehYear                        -0.165
odo_per_year                   -0.162
MMRCurrentRetailCleanPrice     -0.160
VehicleAge                      0.148
MMRCurrentRetailAveragePrice   -0.123
dtype: float64

слабейшие признаки (кандидаты на выброс):
MMRAcquisitionRetailAveragePrice    0.032
model_mean_cost                     0.028
Color_count                        -0.024
VNZIP1_count                       -0.020
cost_to_mmr                         0.010
Trim_count                          0.009
curr_to_acq                        -0.007
IsOnlineSale                        0.000
dtype: float64


In [16]:
# (1) ручной отбор: оставляем фичи с |coef| выше порога
THR = 0.05
keep = coef[coef.abs() > THR].index.tolist()
keep_idx = [feat_names2.index(c) for c in keep]
lr_manual = LogisticRegression(max_iter=2000, random_state=RND).fit(Xtr2[:, keep_idx], ytr)
g_manual = sk_gini(yva, lr_manual.predict_proba(Xva2[:, keep_idx])[:, 1])

# (2) L1: учим логрег с L1 и берём ненулевые веса
lr_l1 = LogisticRegression(penalty="l1", solver="liblinear", C=0.01,
                           max_iter=2000, random_state=RND).fit(Xtr2, ytr)
g_l1 = sk_gini(yva, lr_l1.predict_proba(Xva2)[:, 1])
nonzero = np.sum(lr_l1.coef_[0] != 0)

print(f"все фичи ({len(feat_names2)}):   Gini = {fe_gini['LogisticRegression']:.4f}")
print(f"ручной отбор ({len(keep)}):     Gini = {g_manual:.4f}")
print(f"L1, ненулевых {nonzero}:        Gini = {g_l1:.4f}")

l1_feats = pd.Series(lr_l1.coef_[0], index=feat_names2)
l1_feats = l1_feats[l1_feats != 0].sort_values(key=np.abs, ascending=False)
print("\nфичи, отобранные L1 (топ):")
print(l1_feats.head(10).round(3))

все фичи (38):   Gini = 0.4565
ручной отбор (24):     Gini = 0.4642
L1, ненулевых 13:        Gini = 0.4507

фичи, отобранные L1 (топ):
WheelType_count              -0.370
VehYear                      -0.359
WheelTypeID_count            -0.176
VehOdo                        0.170
VehBCost                     -0.138
Model_count                  -0.097
TopThreeAmericanName_count   -0.075
PRIMEUNIT_count              -0.031
BYRNO_count                  -0.026
Transmission_count            0.020
dtype: float64


- ручной отбор по порогу `|coef|` дал чуть **выше** Gini, чем полный набор - значит часть
  фичей реально были шумом и только мешали;
- L1 с выбранным `C` незначительно ухудшил Gine.

## 9. Подбор гиперпараметров лучшей модели

Лучшая связка - Logistic Regression на расширенном наборе фичей. Перебираю силу
регуляризации `C` (и тип штрафа L1/L2), смотрю на Gini по valid.

In [21]:
results = []
for penalty, solver in [("l2", "lbfgs"), ("l1", "liblinear")]:
    for C in [0.1, 0.5, 1.0, 5.0]:
        m = LogisticRegression(penalty=penalty, solver=solver, C=C,
                               max_iter=3000, random_state=RND).fit(Xtr2[:, keep_idx], ytr)
        results.append((penalty, C, sk_gini(yva, m.predict_proba(Xva2[:, keep_idx])[:, 1])))

res = pd.DataFrame(results, columns=["penalty", "C", "valid_gini"]).sort_values("valid_gini", ascending=False)
print(res.to_string(index=False))

best_row = res.iloc[0]
print(f"\nлучшее: penalty={best_row.penalty}, C={best_row.C}, Gini={best_row.valid_gini:.4f}")

penalty   C  valid_gini
     l2 5.0    0.464354
     l2 1.0    0.464239
     l2 0.5    0.464079
     l2 0.1    0.462791
     l1 1.0    0.460358
     l1 0.5    0.459164
     l1 0.1    0.452123
     l1 5.0    0.326334

лучшее: penalty=l2, C=5.0, Gini=0.4644


**Сильнее всего влияет сила регуляризации `C` (обратная коэффициенту
штрафа).** 

При очень маленьком `C` модель переусредняется и Gini проседает; при разумных
значениях (около 0.1–1.0) качество выходит на плато. Тип штрафа (L1 vs L2) влияет слабее -
в основном на разреженность решения, а не на итоговый Gini. Для KNN ключевым был бы `k`
(число соседей), для GaussianNB настраивать по сути нечего.

In [37]:
# фиксируем лучшую модель
best_penalty = best_row.penalty
best_C = float(best_row.C)
best_solver = "liblinear" if best_penalty == "l1" else "lbfgs"
best_model = LogisticRegression(penalty=best_penalty, solver=best_solver, C=best_C,
                                max_iter=3000, random_state=RND).fit(Xtr2[:, keep_idx], ytr)
print("итоговая модель:", best_model)

итоговая модель: LogisticRegression(C=5.0, max_iter=3000, penalty='l2', random_state=42)


## 10. Gini на train / valid / test и проверка на переобучение

**Переобучена ли модель - разбор.** Смотрим на три числа: train ≈ 0.48, valid ≈ 0.46,
test ≈ 0.27.

- **Train и valid близки** (0.48 vs 0.46). Если бы модель тупо переобучилась под трейн, разрыв между
  train и valid был бы большим. Его нет: у логрега мало параметров (24 фичи на 24k строк) плюс
  регуляризация, классическое переобучение маловероятно.
- **А вот valid→test просадка большая** (0.46 vs 0.27). Это не переобучение под train, а **data drift во времени (concept drift)**: тест - самые поздние покупки (лето–конец 2010), и связь
  между признаками и «лимоном» там частично поплыла относительно начала 2009 года. Глобальная
  линейная граница, настроенная на ранние данные, на поздних ранжирует хуже.

Косвенное подтверждение - в шаге 11 видно, что **KNN на тесте держится заметно лучше** логрега:
локальный алгоритм по соседям устойчивее к такому дрейфу, чем одна глобальная гиперплоскость.

## 11. Precision, Recall, F1, AUC PR - своя реализация

In [26]:
def confusion(y_true, y_pred):
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)
    tp = int(np.sum((y_pred == 1) & (y_true == 1)))
    fp = int(np.sum((y_pred == 1) & (y_true == 0)))
    fn = int(np.sum((y_pred == 0) & (y_true == 1)))
    tn = int(np.sum((y_pred == 0) & (y_true == 0)))
    return tp, fp, fn, tn

def my_precision(y_true, y_pred):
    tp, fp, fn, tn = confusion(y_true, y_pred)
    return tp / (tp + fp) if (tp + fp) else 0.0

def my_recall(y_true, y_pred):
    tp, fp, fn, tn = confusion(y_true, y_pred)
    return tp / (tp + fn) if (tp + fn) else 0.0

def my_f1(y_true, y_pred):
    p, r = my_precision(y_true, y_pred), my_recall(y_true, y_pred)
    return 2 * p * r / (p + r) if (p + r) else 0.0

def my_auc_pr(y_true, scores):
    "Average Precision: сумма precision_k * (recall_k - recall_{k-1})."
    y_true = np.asarray(y_true)
    order = np.argsort(scores)[::-1]         
    y = y_true[order]
    tp = np.cumsum(y == 1)
    fp = np.cumsum(y == 0)
    total_pos = (y_true == 1).sum()
    precision = tp / (tp + fp)
    recall = tp / total_pos
    rec_prev = np.concatenate([[0.0], recall[:-1]])
    return float(np.sum(precision * (recall - rec_prev)))

In [29]:
# сверка с sklearn на valid
proba_v = best_model.predict_proba(Xva2[:, keep_idx])[:, 1]
pred_v = (proba_v >= 0.5).astype(int)
from sklearn.metrics import precision_score, recall_score, f1_score
print("precision: my=%.4f  skl=%.4f" % (my_precision(yva, pred_v), precision_score(yva, pred_v, zero_division=0)))
print("recall   : my=%.4f  skl=%.4f" % (my_recall(yva, pred_v), recall_score(yva, pred_v)))
print("f1       : my=%.4f  skl=%.4f" % (my_f1(yva, pred_v), f1_score(yva, pred_v)))
print("AUC PR   : my=%.4f  skl=%.4f" % (my_auc_pr(yva, proba_v), average_precision_score(yva, proba_v)))

precision: my=0.6017  skl=0.6017
recall   : my=0.1526  skl=0.1526
f1       : my=0.2435  skl=0.2435
AUC PR   : my=0.3549  skl=0.3549


In [33]:
# сравнение трёх алгоритмов на ТЕСТЕ по AUC PR (на расширенных фичах)
final_models = {
    "LogisticRegression": LogisticRegression(penalty=best_penalty, solver=best_solver,
                                             C=best_C, max_iter=3000, random_state=RND),
    "GaussianNB":         GaussianNB(),
    "KNN (k=50)":         KNeighborsClassifier(n_neighbors=50),
}
print(f"{'модель':22s}{'AUC PR':>9s}{'Gini':>9s}")
for name, model in final_models.items():
    model.fit(Xtr2, ytr)
    proba_t = model.predict_proba(Xte2)[:, 1]
    print(f"{name:22s}{my_auc_pr(yte, proba_t):>9.4f}{my_gini(yte, proba_t):>9.4f}")

print(f"\nбазовая частота положительного класса на тесте: {yte.mean():.4f}")

модель                   AUC PR     Gini
LogisticRegression       0.1601   0.2666
GaussianNB               0.1610   0.2771
KNN (k=50)               0.3969   0.4156

базовая частота положительного класса на тесте: 0.1237


По AUC PR (как и по Gini) **лидирует KNN**, а логрег
с наивным Байесом проседают. Линейная граница, идеально
настроенная на valid, на позднем тесте ранжирует хуже, а KNN со своими локальными соседями
устойчивее. 

Baseline для AUC PR - это доля положительного класса (примерно 0.12); все модели выше неё,
то есть «лимоны» ловят лучше случайного, но запас у KNN на тесте ощутимо больше.

## 12. Какую hard-label метрику выбрать для поиска «лимонов»

Класс «лимонов» редкий (примерно 12%), поэтому **accuracy не годится**: модель, которая всегда
говорит «хорошая машина», даст примерно 88% точности и при этом не поймает ни одного лимона. Толку ноль.

Дальше вопрос цены ошибок:
- **FN** (купили лимон, не распознали) - прямой убыток: ремонт/перепродажа в минус;
- **FP** (хорошую машину назвали лимоном) - упущенная выгодная покупка.

Для аукционного покупателя обе ошибки стоят денег, и заранее перекоса в сторону только recall
или только precision нет. Поэтому как итоговую жёсткую метрику беру **F1** - гармоническое
среднее precision и recall: она штрафует за перекос в любую сторону и адекватно работает на
несбалансированных классах.

Если же в бизнес задаче пропустить лимон гораздо дороже, чем перестраховаться, - стоит
сместиться в сторону recall и взять **F-beta с beta > 1**, которая даёт recall
больший вес.